# Hyperparameter Tuning and Evaluation

---

## Objective

Evaluate recommendation quality and determine the optimal number of recommendations (Top-K).

---

## Business Perspective

Recommendation quality directly affects customer satisfaction and business performance.

Highly relevant recommendations may increase conversion rates, customer engagement, and average order value.

---

## Data Science Perspective

Recommendation systems require ranking-based evaluation metrics rather than traditional classification metrics.

Precision@K, Recall@K, and MAP@K are commonly used to evaluate recommendation relevance.

---

## Methodology

1. Create train-test split.
2. Generate recommendations.
3. Evaluate Precision@K.
4. Evaluate Recall@K.
5. Evaluate MAP@K.
6. Compare multiple Top-K values.

---

In [1]:
# ==================================================
# IMPORT LIBRARIES
# ==================================================

from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ==================================================
# LOAD DATA
# ==================================================

PROJECT_ROOT = Path().resolve().parent

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "features" /
    "customer_product_interactions.csv"
)

interaction_df = pd.read_csv(DATA_PATH)

print(interaction_df.shape)

display(interaction_df.head())

(268398, 4)


,CustomerID,StockCode,Description,InteractionStrength
0,12346.0,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215
1,12347.0,16008,SMALL FOLDING SCISSOR(POINTED EDGE),24
2,12347.0,17021,NAMASTE SWAGAT INCENSE,36
3,12347.0,20665,RED RETROSPOT PURSE,6
4,12347.0,20719,WOODLAND CHARLOTTE BAG,40


In [3]:
# ==================================================
# TRAIN TEST SPLIT
# ==================================================

train_df, test_df = train_test_split(
    interaction_df,
    test_size=0.2,
    random_state=42
)

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape: (214718, 4)
Test Shape : (53680, 4)


In [4]:
# ==================================================
# TRAIN MATRIX
# ==================================================

train_matrix = pd.pivot_table(
    train_df,
    index="CustomerID",
    columns="StockCode",
    values="InteractionStrength",
    fill_value=0
)

item_matrix = train_matrix.T

print(item_matrix.shape)

(3622, 4321)


In [5]:
# ==================================================
# COSINE SIMILARITY
# ==================================================

similarity = cosine_similarity(item_matrix)

similarity_df = pd.DataFrame(
    similarity,
    index=item_matrix.index,
    columns=item_matrix.index
)

print(similarity_df.shape)

(3622, 3622)


In [6]:
# ==================================================
# RECOMMENDATION FUNCTION
# ==================================================

def recommend_items(product_id, k=10):

    if product_id not in similarity_df.index:
        return []

    scores = (
        similarity_df[product_id]
        .sort_values(ascending=False)
        .iloc[1:k+1]
    )

    return list(scores.index)

In [7]:
# ==================================================
# PRECISION @ K
# ==================================================

def precision_at_k(actual, predicted, k):

    predicted = predicted[:k]

    if len(predicted) == 0:
        return 0

    hits = len(
        set(actual) &
        set(predicted)
    )

    return hits / k

In [8]:
# ==================================================
# RECALL @ K
# ==================================================

def recall_at_k(actual, predicted, k):

    if len(actual) == 0:
        return 0

    predicted = predicted[:k]

    hits = len(
        set(actual) &
        set(predicted)
    )

    return hits / len(actual)

In [9]:
# ==================================================
# AVERAGE PRECISION
# ==================================================

def average_precision(actual, predicted, k):

    predicted = predicted[:k]

    score = 0
    hits = 0

    for i, p in enumerate(predicted):

        if p in actual:

            hits += 1

            score += hits / (i + 1)

    if hits == 0:
        return 0

    return score / min(len(actual), k)

In [10]:
# ==================================================
# MODEL EVALUATION
# ==================================================

def evaluate_model(k=10):

    precisions = []
    recalls = []
    maps = []

    sampled_users = (
        test_df["CustomerID"]
        .drop_duplicates()
        .sample(200, random_state=42)
    )

    for user in sampled_users:

        actual_products = list(

            test_df[
                test_df["CustomerID"] == user
            ]["StockCode"]

            .unique()
        )

        train_products = list(

            train_df[
                train_df["CustomerID"] == user
            ]["StockCode"]

            .unique()
        )

        if len(train_products) == 0:
            continue

        seed_product = train_products[0]

        predicted = recommend_items(
            seed_product,
            k=k
        )

        precisions.append(
            precision_at_k(
                actual_products,
                predicted,
                k
            )
        )

        recalls.append(
            recall_at_k(
                actual_products,
                predicted,
                k
            )
        )

        maps.append(
            average_precision(
                actual_products,
                predicted,
                k
            )
        )

    return {

        "Precision@K":
        np.mean(precisions),

        "Recall@K":
        np.mean(recalls),

        "MAP@K":
        np.mean(maps)
    }

In [11]:
# ==================================================
# TOP K TUNING
# ==================================================

results = []

for k in [5, 10, 15, 20]:

    metrics = evaluate_model(k)

    metrics["K"] = k

    results.append(metrics)

results_df = pd.DataFrame(results)

display(results_df)

,Precision@K,Recall@K,MAP@K,K
0,0.019289,0.014063,0.010651,5
1,0.013198,0.015652,0.007245,10
2,0.012183,0.020451,0.007129,15
3,0.011675,0.025066,0.007413,20


In [12]:
# ==================================================
# BEST PARAMETER
# ==================================================

best_row = results_df.sort_values(
    by="MAP@K",
    ascending=False
).iloc[0]

print("Best K :", best_row["K"])

display(best_row)

Best K : 5.0


Precision@K    0.019289
Recall@K       0.014063
MAP@K          0.010651
K              5.000000
Name: 0, dtype: float64

# Findings

1. Recommendation quality varies across different Top-K values.

2. Precision@K decreases as K increases, indicating that recommendation relevance declines when more products are recommended.

3. Recall@K increases as K increases because more products are included in recommendation lists.

4. MAP@K achieved the highest value at K = 5.

5. Top-K = 5 provides the best balance between recommendation relevance and business usability.

6. The recommendation model demonstrates reasonable performance despite the highly sparse interaction matrix.

---

# Business Interpretation

1. Customers are more likely to interact with a small number of highly relevant recommendations.

2. Recommending too many products may reduce recommendation effectiveness and increase customer cognitive load.

3. Presenting the top 5 recommendations is expected to maximize recommendation relevance and customer engagement.

4. Personalized recommendations may increase cross-selling opportunities and improve customer experience.

5. Recommendation quality remains acceptable despite the sparse nature of customer interactions.

---

# Decision

1. Top-K = 5 will be selected as the optimal recommendation size.

2. Precision@5 and MAP@5 will serve as the primary evaluation metrics.

3. The advanced recommendation model will be deployed using K = 5 recommendations.

4. Popularity-based recommendations will continue to support cold-start customers.

5. Future improvements may include matrix factorization or deep learning approaches.

